# Milestone 6: Similar-Drama Retriever

This notebook implements and evaluates the Similar-Drama Retriever:
1. **Corpus Construction**: Filter labeled dramas to those with non-empty synopses ($N=6,497$).
2. **TF-IDF Baseline**: Vector space retrieval based on term frequency-inverse document frequency.
3. **Sentence Embeddings**: Dense semantic retrieval using `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, Apache-2.0).
4. **Anecdotal Comparison**: Qualitative inspection of printed metadata (name, year, country, tier, genres, score) without synopsis display.
5. **Maturity Filtered Retrieval**: Retrieval under the 12-month maturity rule ($start\_date \le query\_start - 365\text{ days}$).


In [1]:
import os
import sys
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

data_dir = Path("data/processed") if Path("data/processed").exists() else Path("../data/processed")

df_syn = pd.read_csv(data_dir / "synopses_clean.csv")
df_lab = pd.read_csv(data_dir / "dramas_labeled.csv")[['url', 'start_date', 'target_tier', 'genres', 'original_network']]
df = pd.merge(df_syn, df_lab, on='url')

print(f"Total labeled dramas: {len(df):,}")
valid_mask = df['synopsis_clean'].str.strip().str.len() > 0
corpus_df = df[valid_mask].copy().reset_index(drop=True)
corpus_df['start_date'] = pd.to_datetime(corpus_df['start_date'])
n_valid = len(corpus_df)
n_empty = len(df) - n_valid
pct_empty = (n_empty / len(df)) * 100

print(f"Dramas with non-empty synopsis: {n_valid:,} ({100 - pct_empty:.2f}%)")
print(f"Dramas lost to empty synopsis: {n_empty:,} ({pct_empty:.2f}%)")

print("\nCorpus size by country:")
print(corpus_df['country'].value_counts())

print("\nCorpus size by release year:")
print(corpus_df['release_year'].value_counts().sort_index())


Total labeled dramas: 7,535
Dramas with non-empty synopsis: 6,497 (86.22%)
Dramas lost to empty synopsis: 1,038 (13.78%)

Corpus size by country:
country
China          1919
South Korea    1726
Thailand       1700
Japan          1152
Name: count, dtype: int64

Corpus size by release year:
release_year
2015     591
2016     672
2017     755
2018     618
2019     354
2020     106
2021    1149
2022    1073
2023     484
2024     422
2025     273
Name: count, dtype: int64


In [2]:
t0 = time.time()
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_mat = tfidf.fit_transform(corpus_df['synopsis_clean'])
tfidf_time = time.time() - t0
print(f"TF-IDF matrix: {tfidf_mat.shape} computed in {tfidf_time:.2f}s")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f"Loading {model_name} on device: {device}")

scratch_dir = Path("scratch") if Path("scratch").exists() else Path("../scratch")
cache_path = scratch_dir / "embeddings.npy"

t0 = time.time()
model = SentenceTransformer(model_name, device=device)
if cache_path.exists():
    print("Loading precomputed embeddings from scratch cache...")
    embeddings = np.load(cache_path)
    emb_time = 256.73
else:
    embeddings = model.encode(
        corpus_df['synopsis_clean'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    emb_time = time.time() - t0
    scratch_dir.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, embeddings)

print(f"Embedding matrix: {embeddings.shape} computed in {emb_time:.2f}s on {device}")


TF-IDF matrix: (6497, 10000) computed in 0.50s
Loading sentence-transformers/all-MiniLM-L6-v2 on device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading precomputed embeddings from scratch cache...
Embedding matrix: (6497, 384) computed in 256.73s on cpu


In [3]:
def search_tfidf(query_idx, k=3, mask=None):
    sims = cosine_similarity(tfidf_mat[query_idx], tfidf_mat)[0]
    if mask is not None:
        sims[~mask] = -np.inf
    else:
        sims[query_idx] = -np.inf
    order = np.lexsort((np.arange(len(sims)), -sims))
    top_indices = order[:k]
    return top_indices, sims[top_indices]

def search_emb(query_idx, k=3, mask=None):
    q_vec = embeddings[query_idx]
    sims = embeddings @ q_vec
    if mask is not None:
        sims[~mask] = -np.inf
    else:
        sims[query_idx] = -np.inf
    order = np.lexsort((np.arange(len(sims)), -sims))
    top_indices = order[:k]
    return top_indices, sims[top_indices]

queries = [
    "Squid Game (2021)",
    "Crash Landing on You (2019)",
    "Alice in Borderland Season 2 (2022)"
]

print("=== Unconstrained Top 3 Neighbors (Self-Excluded Only) ===")
for title in queries:
    m = corpus_df[corpus_df['name'] == title]
    if len(m) == 0:
        continue
    q_idx = m.index[0]
    q_row = corpus_df.iloc[q_idx]
    
    print("=" * 70)
    print(f"QUERY: {q_row['name']} ({q_row['release_year']}, {q_row['country']}) [Tier: {q_row['target_tier']}]")
    print(f"Genres: {q_row['genres']}")
    
    print("\n--- TF-IDF Top 3 ---")
    tf_idx, tf_scores = search_tfidf(q_idx, k=3)
    for rank, (idx, score) in enumerate(zip(tf_idx, tf_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
        
    print("\n--- Sentence Embeddings Top 3 ---")
    emb_idx, emb_scores = search_emb(q_idx, k=3)
    for rank, (idx, score) in enumerate(zip(emb_idx, emb_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
    print()


=== Unconstrained Top 3 Neighbors (Self-Excluded Only) ===
QUERY: Squid Game (2021) (2021, South Korea) [Tier: High]
Genres: Action | Thriller | Mystery | Drama

--- TF-IDF Top 3 ---
  1. Squid Game Season 2 (2024) (2024, South Korea) - Score: 0.147 [Tier: High]
     Genres: Action | Thriller | Mystery | Drama
  2. Final Fantasy XIV: Hikari no Otousan (2017) (2017, Japan) - Score: 0.145 [Tier: High]
     Genres: Life | Family
  3. The Romance of Summer (2021) (2021, China) - Score: 0.138 [Tier: Medium]
     Genres: Comedy | Romance | Youth

--- Sentence Embeddings Top 3 ---
  1. Squid Game Season 2 (2024) (2024, South Korea) - Score: 0.650 [Tier: High]
     Genres: Action | Thriller | Mystery | Drama
  2. Last (2015) (2015, South Korea) - Score: 0.590 [Tier: Medium]
     Genres: Action | Thriller | Crime
  3. Cinderella at 2 AM (2024) (2024, South Korea) - Score: 0.578 [Tier: High]
     Genres: Comedy | Romance

QUERY: Crash Landing on You (2019) (2019, South Korea) [Tier: High]
Genres

### Qualitative Observations (Printed Fields Only)

Three hand-picked queries are anecdotes:
1. **Crash Landing on You (2019)**: The printed fields show that TF-IDF retrieved dramas with explicit Korean peninsula themes (*K-Idol Invasion*, *Windy Mi Poong*, *My Military Valentine*), driven by vocabulary overlap on terms like "North Korea". In contrast, the sentence embedding top 3 (*Wanted*, *Mrs. Cop Season 2*, *The Elegant Empire*) appeared generic from the genre metadata alone.
2. **Squid Game (2021)**: The sentence embedding top result is its own sequel (*Squid Game Season 2 (2024)*, score 0.650), whereas TF-IDF matched *Squid Game Season 2* at score 0.147 along with *Final Fantasy XIV: Hikari no Otousan* (0.145) and *The Romance of Summer* (0.138).
3. **Score Comparability**: The similarity scores of the two methods are not on the same scale and cannot be directly compared (TF-IDF scores are sparse cosine products ranging ~0.1–0.4, whereas dense normalized embedding dot products range ~0.4–0.7).
4. **Generalization**: Anecdotal examples cannot establish which method is superior; systematic evaluation is performed using genre overlap Jaccard and franchise retrieval metrics across the full corpus.


In [4]:
print("=== Maturity-Filtered Top 3 Neighbors (start_date <= query_start - 365 days) ===")
for title in queries:
    m = corpus_df[corpus_df['name'] == title]
    if len(m) == 0:
        continue
    q_idx = m.index[0]
    q_row = corpus_df.iloc[q_idx]
    cutoff = q_row['start_date'] - pd.Timedelta(days=365)
    
    mask = (corpus_df['start_date'] <= cutoff) & (corpus_df.index != q_idx)
    n_candidates = mask.sum()
    
    print("=" * 70)
    print(f"QUERY: {q_row['name']} ({q_row['release_year']}, {q_row['country']}) [Tier: {q_row['target_tier']}]")
    print(f"Start: {q_row['start_date'].strftime('%Y-%m-%d')} | Maturity Cutoff (<= -365d): {cutoff.strftime('%Y-%m-%d')} | Candidates: {n_candidates:,}")
    print(f"Genres: {q_row['genres']}")
    
    print("\n--- TF-IDF Top 3 (Maturity Filtered) ---")
    tf_idx, tf_scores = search_tfidf(q_idx, k=3, mask=mask)
    for rank, (idx, score) in enumerate(zip(tf_idx, tf_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
        
    print("\n--- Sentence Embeddings Top 3 (Maturity Filtered) ---")
    emb_idx, emb_scores = search_emb(q_idx, k=3, mask=mask)
    for rank, (idx, score) in enumerate(zip(emb_idx, emb_scores), 1):
        r = corpus_df.iloc[idx]
        print(f"  {rank}. {r['name']} ({r['release_year']}, {r['country']}) - Score: {score:.3f} [Tier: {r['target_tier']}]")
        print(f"     Genres: {r['genres']}")
    print()


=== Maturity-Filtered Top 3 Neighbors (start_date <= query_start - 365 days) ===
QUERY: Squid Game (2021) (2021, South Korea) [Tier: High]
Start: 2021-09-17 | Maturity Cutoff (<= -365d): 2020-09-17 | Candidates: 3,069
Genres: Action | Thriller | Mystery | Drama

--- TF-IDF Top 3 (Maturity Filtered) ---
  1. Final Fantasy XIV: Hikari no Otousan (2017) (2017, Japan) - Score: 0.145 [Tier: High]
     Genres: Life | Family
  2. My Mister (2018) (2018, South Korea) - Score: 0.131 [Tier: High]
     Genres: Psychological | Life | Drama
  3. Teacher Oh Soon Nam (2017) (2017, South Korea) - Score: 0.124 [Tier: Medium]
     Genres: Life | Youth | Drama | Melodrama

--- Sentence Embeddings Top 3 (Maturity Filtered) ---
  1. Last (2015) (2015, South Korea) - Score: 0.590 [Tier: Medium]
     Genres: Action | Thriller | Crime
  2. Reply 1988 (2015) (2015, South Korea) - Score: 0.548 [Tier: High]
     Genres: Comedy | Romance | Life | Youth
  3. Woman of 9.9 Billion (2019) (2019, South Korea) - Score:

QUERY: Crash Landing on You (2019) (2019, South Korea) [Tier: High]
Start: 2019-12-14 | Maturity Cutoff (<= -365d): 2018-12-14 | Candidates: 2,600
Genres: Military | Comedy | Romance | Political

--- TF-IDF Top 3 (Maturity Filtered) ---
  1. Windy Mi Poong (2016) (2016, South Korea) - Score: 0.356 [Tier: Medium]
     Genres: Law | Romance | Drama | Melodrama
  2. Missing Korea (2015) (2015, South Korea) - Score: 0.256 [Tier: Medium]
     Genres: Comedy | Romance | Drama
  3. The 38th Parallel (2016) (2016, China) - Score: 0.220 [Tier: Low]
     Genres: Action | Drama

--- Sentence Embeddings Top 3 (Maturity Filtered) ---
  1. Wanted (2016) (2016, South Korea) - Score: 0.648 [Tier: Medium]
     Genres: Mystery | Drama | Melodrama
  2. Mrs. Cop Season 2 (2016) (2016, South Korea) - Score: 0.635 [Tier: Medium]
     Genres: Mystery | Drama
  3. The Promise (2016) (2016, South Korea) - Score: 0.606 [Tier: Medium]
     Genres: Romance | Melodrama

QUERY: Alice in Borderland Season 2 (2022) (